# WESAD - Pipeline Completo de Ciencia de Datos
## Detección de Estrés mediante Señales Fisiológicas Wearable

**Dataset:** [WESAD (Wearable Stress and Affect Detection)](https://www.kaggle.com/datasets/orvile/wesad-wearable-stress-affect-detection-dataset)  
**Referencia:** Schmidt, Philip & Reiss, Attila et al. (2018). *Introducing WESAD, a Multimodal Dataset for Wearable Stress and Affect Detection.* ICMI 2018.  
**Código base:** [Kaggle - WESAD Stress Class](https://www.kaggle.com/code/apurvpanchal/wesad-stress-class)

---

### Contenido del Notebook

| # | Actividad | Descripción |
|---|-----------|-------------|
| 1 | **Extracción de Características** | Extraer features de las señales fisiológicas y etiquetar los datos |
| 2 | **Análisis Exploratorio (EDA)** | Exploración estadística y visual del dataset resultante |
| 3 | **Preprocesamiento** | Limpieza, imputación, eliminación de redundancias y estandarización |
| 4 | **Ranking con Factor de Fisher** | Rankear características según su capacidad discriminativa |
| 5 | **Selección Escalar Hacia Adelante** | Seleccionar las 5 mejores características |

---

### Estructura del Dataset WESAD

- **15 sujetos** (S2–S17, sin S1 ni S12)
- **Archivos:** `.pkl` por sujeto con señales de chest (RespiBAN) y wrist (Empatica E4)
- **Etiquetas:** 0 = no definido, 1 = baseline, 2 = stress, 3 = amusement, 4 = meditation
- **Señales chest (700 Hz):** ACC (3 ejes), ECG, EDA, EMG, Respiración, Temperatura
- **Señales wrist:** BVP (64 Hz), EDA (4 Hz), TEMP (4 Hz), ACC (32 Hz)

## 0. Importación de Librerías y Configuración

In [1]:
import numpy as np
import pandas as pd
import pickle
import os
import json
import warnings
from scipy import stats, signal
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
np.random.seed(42)

print('Librerías importadas correctamente.')

Librerías importadas correctamente.


c:\Users\frida\AppData\Local\pypoetry\Cache\virtualenvs\recocimiento_de_patrones-3a7zzQ48-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configuración de Rutas y Parámetros

> **⚠️ IMPORTANTE:** Modifica `DATA_PATH` con la ruta donde descargaste el dataset WESAD de Kaggle.  
> La estructura esperada es: `DATA_PATH/S2/S2.pkl`, `DATA_PATH/S3/S3.pkl`, etc.

In [2]:
# ═══════════════════════════════════════════════════════════════════
# CONFIGURACIÓN - MODIFICAR SEGÚN TU ENTORNO
# ═══════════════════════════════════════════════════════════════════

# Cambiar esta ruta a donde tengas el dataset WESAD descargado de Kaggle
# Descargar dataset
path = kagglehub.dataset_download(
    "orvile/wesad-wearable-stress-affect-detection-dataset"
)

print("Path raíz:", path)
print("Contenido raíz:", os.listdir(path))

# Ruta correcta al dataset WESAD
DATA_PATH = os.path.join(path, "WESAD")

# IDs de los 15 sujetos del estudio
SUBJECT_IDS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

# Directorio de salida para resultados
OUTPUT_DIR = './resultados/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Frecuencias de muestreo de los sensores
FS_CHEST = 700       # Hz - RespiBAN (chest)
FS_BVP = 64          # Hz - Empatica E4 BVP
FS_EDA_WRIST = 4     # Hz - Empatica E4 EDA
FS_TEMP_WRIST = 4    # Hz - Empatica E4 TEMP
FS_ACC_WRIST = 32    # Hz - Empatica E4 ACC

# Parámetros de ventaneo para extracción de características
WINDOW_SIZE = 60     # segundos
WINDOW_SHIFT = 30    # segundos (50% overlap)

# Mapeo de clases: binario (stress vs no-stress)
# baseline(1) y amusement(3) → 0 (no-stress)
# stress(2) → 1 (stress)
LABEL_MAP = {1: 0, 2: 1, 3: 0}

print(f'Ruta del dataset: {DATA_PATH}')
print(f'Sujetos: {SUBJECT_IDS}')
print(f'Ventana: {WINDOW_SIZE}s con desplazamiento de {WINDOW_SHIFT}s')

Path raíz: C:\Users\frida\.cache\kagglehub\datasets\orvile\wesad-wearable-stress-affect-detection-dataset\versions\1
Contenido raíz: ['WESAD']
Ruta del dataset: C:\Users\frida\.cache\kagglehub\datasets\orvile\wesad-wearable-stress-affect-detection-dataset\versions\1\WESAD
Sujetos: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
Ventana: 60s con desplazamiento de 30s


---
## Actividad 1: Extracción de Características de Señales y Etiquetado

En esta sección se implementa la extracción de características (features) de las señales fisiológicas del dataset WESAD, siguiendo el enfoque del notebook de referencia en Kaggle.

### Proceso:
1. Cargar los archivos `.pkl` de cada sujeto
2. Segmentar las señales en ventanas de 60 segundos con 50% de solapamiento
3. Para cada ventana, extraer características estadísticas y frecuenciales de cada modalidad de señal
4. Asignar la etiqueta por voto mayoritario dentro de la ventana

### Características extraídas por señal:
- **Estadísticas:** media, desv. estándar, mín, máx, mediana, rango, curtosis, asimetría, percentiles, RMS, tasa de cruces por cero
- **Frecuenciales:** frecuencia dominante, energía espectral, entropía espectral
- **ECG específicas:** frecuencia cardíaca, intervalos RR, SDNN, RMSSD, pNN50
- **EDA específicas:** componente tónica (SCL), componente fásica (SCR), número de picos SCR
- **Respiración:** tasa respiratoria media
- **Temperatura:** pendiente (slope), derivada
- **EMG:** amplitud media absoluta (MAV), varianza, longitud de forma de onda

### 1.1 Función de Carga de Datos

In [3]:
def load_subject_data(data_path, subject_id):
    """
    Carga los datos de un sujeto desde el archivo .pkl de WESAD.
    
    Estructura del .pkl:
    - 'signal' -> 'chest' -> {'ACC','ECG','EDA','EMG','Resp','Temp'}
    - 'signal' -> 'wrist' -> {'ACC','BVP','EDA','TEMP'}
    - 'label' -> array de etiquetas (muestreadas a 700Hz)
    """
    subject_str = f'S{subject_id}'
    pkl_path = os.path.join(data_path, subject_str, f'{subject_str}.pkl')
    
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    
    return data

### 1.2 Funciones de Extracción de Características

In [4]:
def compute_statistical_features(signal_data, prefix):
    """
    Calcula características estadísticas básicas de una señal.
    
    Parámetros:
    - signal_data: array numpy con los datos de la señal
    - prefix: prefijo para los nombres de las características
    
    Retorna:
    - dict con las características calculadas
    """
    features = {}
    if len(signal_data) == 0:
        return features
    
    if signal_data.ndim > 1:
        signal_data = signal_data.flatten()
    
    features[f'{prefix}_mean'] = np.mean(signal_data)
    features[f'{prefix}_std'] = np.std(signal_data)
    features[f'{prefix}_min'] = np.min(signal_data)
    features[f'{prefix}_max'] = np.max(signal_data)
    features[f'{prefix}_median'] = np.median(signal_data)
    features[f'{prefix}_range'] = np.max(signal_data) - np.min(signal_data)
    features[f'{prefix}_kurtosis'] = stats.kurtosis(signal_data)
    features[f'{prefix}_skewness'] = stats.skew(signal_data)
    features[f'{prefix}_q25'] = np.percentile(signal_data, 25)
    features[f'{prefix}_q75'] = np.percentile(signal_data, 75)
    features[f'{prefix}_iqr'] = features[f'{prefix}_q75'] - features[f'{prefix}_q25']
    features[f'{prefix}_rms'] = np.sqrt(np.mean(signal_data ** 2))
    
    # Tasa de cruces por cero
    zero_crossings = np.sum(np.diff(np.sign(signal_data - np.mean(signal_data))) != 0)
    features[f'{prefix}_zcr'] = zero_crossings / len(signal_data)
    
    return features


def compute_frequency_features(signal_data, fs, prefix):
    """
    Calcula características en el dominio de la frecuencia usando FFT.
    """
    features = {}
    if len(signal_data) < 4:
        return features
    
    if signal_data.ndim > 1:
        signal_data = signal_data.flatten()
    
    N = len(signal_data)
    yf = np.abs(fft(signal_data - np.mean(signal_data)))[:N // 2]
    xf = fftfreq(N, 1 / fs)[:N // 2]
    
    if len(yf) > 0 and np.sum(yf) > 0:
        features[f'{prefix}_peak_freq'] = xf[np.argmax(yf)]
        features[f'{prefix}_spectral_energy'] = np.sum(yf ** 2) / N
        psd = yf ** 2 / np.sum(yf ** 2)
        psd = psd[psd > 0]
        features[f'{prefix}_spectral_entropy'] = -np.sum(psd * np.log2(psd))
    
    return features

print('Funciones estadísticas y frecuenciales definidas.')

Funciones estadísticas y frecuenciales definidas.


### 1.3 Funciones Específicas por Modalidad de Señal

In [5]:
def compute_eda_features(eda_signal, fs, prefix='EDA'):
    """
    Características de EDA (Actividad Electrodérmica).
    Incluye separación tónica (SCL) / fásica (SCR) simplificada.
    """
    features = compute_statistical_features(eda_signal, prefix)
    features.update(compute_frequency_features(eda_signal, fs, prefix))
    
    if eda_signal.ndim > 1:
        eda_signal = eda_signal.flatten()
    
    if len(eda_signal) > 10:
        try:
            # Componente tónica (SCL) - filtro paso bajo
            b, a = signal.butter(2, 0.05 / (fs / 2), btype='low')
            scl = signal.filtfilt(b, a, eda_signal)
            features[f'{prefix}_scl_mean'] = np.mean(scl)
            features[f'{prefix}_scl_std'] = np.std(scl)
            
            # Componente fásica (SCR)
            scr = eda_signal - scl
            features[f'{prefix}_scr_mean'] = np.mean(scr)
            features[f'{prefix}_scr_std'] = np.std(scr)
            
            # Número de picos SCR
            peaks, _ = signal.find_peaks(scr, height=np.std(scr) * 0.5)
            features[f'{prefix}_scr_num_peaks'] = len(peaks)
        except Exception:
            pass
    
    if len(eda_signal) > 1:
        eda_deriv = np.diff(eda_signal)
        features[f'{prefix}_deriv_mean'] = np.mean(eda_deriv)
        features[f'{prefix}_deriv_std'] = np.std(eda_deriv)
    
    return features


def compute_ecg_features(ecg_signal, fs=700, prefix='ECG'):
    """
    Características de ECG incluyendo HRV (Variabilidad de Frecuencia Cardíaca).
    """
    features = compute_statistical_features(ecg_signal, prefix)
    features.update(compute_frequency_features(ecg_signal, fs, prefix))
    
    if ecg_signal.ndim > 1:
        ecg_signal = ecg_signal.flatten()
    
    try:
        # Filtro paso banda para ECG (0.5 - 40 Hz)
        b, a = signal.butter(4, [0.5 / (fs / 2), 40 / (fs / 2)], btype='band')
        ecg_filtered = signal.filtfilt(b, a, ecg_signal)
        
        # Detección de picos R
        min_distance = int(0.5 * fs)
        peaks, _ = signal.find_peaks(
            ecg_filtered,
            height=np.mean(ecg_filtered) + 0.5 * np.std(ecg_filtered),
            distance=min_distance
        )
        
        if len(peaks) > 2:
            rr_intervals = np.diff(peaks) / fs  # en segundos
            features[f'{prefix}_hr_mean'] = 60.0 / np.mean(rr_intervals)
            features[f'{prefix}_hr_std'] = np.std(60.0 / rr_intervals)
            features[f'{prefix}_rr_mean'] = np.mean(rr_intervals)
            features[f'{prefix}_rr_std'] = np.std(rr_intervals)   # SDNN
            features[f'{prefix}_rmssd'] = np.sqrt(np.mean(np.diff(rr_intervals) ** 2))
            nn50 = np.sum(np.abs(np.diff(rr_intervals)) > 0.05)
            features[f'{prefix}_pnn50'] = nn50 / len(rr_intervals)
        else:
            features[f'{prefix}_hr_mean'] = 0
            features[f'{prefix}_hr_std'] = 0
    except Exception:
        pass
    
    return features


def compute_acc_features(acc_data, fs, prefix='ACC'):
    """Características del acelerómetro (3 ejes + magnitud)."""
    features = {}
    if acc_data.ndim == 1:
        acc_data = acc_data.reshape(-1, 1)
    
    axes = ['x', 'y', 'z'] if acc_data.shape[1] >= 3 else [str(i) for i in range(acc_data.shape[1])]
    for i, axis in enumerate(axes[:acc_data.shape[1]]):
        features.update(compute_statistical_features(acc_data[:, i], f'{prefix}_{axis}'))
    
    if acc_data.shape[1] >= 3:
        magnitude = np.sqrt(np.sum(acc_data[:, :3] ** 2, axis=1))
        features.update(compute_statistical_features(magnitude, f'{prefix}_mag'))
        features.update(compute_frequency_features(magnitude, fs, f'{prefix}_mag'))
    
    return features


def compute_resp_features(resp_signal, fs=700, prefix='RESP'):
    """Características de la señal de respiración."""
    features = compute_statistical_features(resp_signal, prefix)
    features.update(compute_frequency_features(resp_signal, fs, prefix))
    
    if resp_signal.ndim > 1:
        resp_signal = resp_signal.flatten()
    
    try:
        peaks, _ = signal.find_peaks(resp_signal, distance=int(fs * 1.5))
        if len(peaks) > 1:
            breath_intervals = np.diff(peaks) / fs
            features[f'{prefix}_rate_mean'] = 60.0 / np.mean(breath_intervals)
            features[f'{prefix}_rate_std'] = np.std(60.0 / breath_intervals)
            features[f'{prefix}_insp_time'] = np.mean(breath_intervals)
    except Exception:
        pass
    
    return features


def compute_temp_features(temp_signal, fs, prefix='TEMP'):
    """Características de temperatura."""
    features = compute_statistical_features(temp_signal, prefix)
    
    if temp_signal.ndim > 1:
        temp_signal = temp_signal.flatten()
    
    if len(temp_signal) > 1:
        x = np.arange(len(temp_signal))
        slope, _, _, _, _ = stats.linregress(x, temp_signal)
        features[f'{prefix}_slope'] = slope
        temp_deriv = np.gradient(temp_signal)
        features[f'{prefix}_deriv_mean'] = np.mean(temp_deriv)
        features[f'{prefix}_deriv_std'] = np.std(temp_deriv)
    
    return features


def compute_emg_features(emg_signal, fs=700, prefix='EMG'):
    """Características de EMG (Electromiograma)."""
    features = compute_statistical_features(emg_signal, prefix)
    features.update(compute_frequency_features(emg_signal, fs, prefix))
    
    if emg_signal.ndim > 1:
        emg_signal = emg_signal.flatten()
    
    features[f'{prefix}_mav'] = np.mean(np.abs(emg_signal))
    features[f'{prefix}_var'] = np.var(emg_signal)
    if len(emg_signal) > 1:
        features[f'{prefix}_wl'] = np.sum(np.abs(np.diff(emg_signal)))
    
    return features

print('Funciones de extracción por modalidad definidas.')

Funciones de extracción por modalidad definidas.


### 1.4 Extracción de Características por Ventana y por Sujeto

In [6]:
def extract_features_window(chest_data, wrist_data, labels_window):
    """
    Extrae todas las características de una ventana temporal.
    Combina señales de chest (RespiBAN) y wrist (Empatica E4).
    """
    all_features = {}
    
    # ═══ SEÑALES DEL CHEST (RespiBAN - 700 Hz) ═══
    if 'ECG' in chest_data and len(chest_data['ECG']) > 0:
        all_features.update(compute_ecg_features(chest_data['ECG'], FS_CHEST, 'c_ECG'))
    if 'EDA' in chest_data and len(chest_data['EDA']) > 0:
        all_features.update(compute_eda_features(chest_data['EDA'], FS_CHEST, 'c_EDA'))
    if 'EMG' in chest_data and len(chest_data['EMG']) > 0:
        all_features.update(compute_emg_features(chest_data['EMG'], FS_CHEST, 'c_EMG'))
    if 'Resp' in chest_data and len(chest_data['Resp']) > 0:
        all_features.update(compute_resp_features(chest_data['Resp'], FS_CHEST, 'c_RESP'))
    if 'Temp' in chest_data and len(chest_data['Temp']) > 0:
        all_features.update(compute_temp_features(chest_data['Temp'], FS_CHEST, 'c_TEMP'))
    if 'ACC' in chest_data and len(chest_data['ACC']) > 0:
        all_features.update(compute_acc_features(chest_data['ACC'], FS_CHEST, 'c_ACC'))
    
    # ═══ SEÑALES DEL WRIST (Empatica E4) ═══
    if 'BVP' in wrist_data and len(wrist_data['BVP']) > 0:
        all_features.update(compute_statistical_features(wrist_data['BVP'], 'w_BVP'))
        all_features.update(compute_frequency_features(wrist_data['BVP'], FS_BVP, 'w_BVP'))
    if 'EDA' in wrist_data and len(wrist_data['EDA']) > 0:
        all_features.update(compute_eda_features(wrist_data['EDA'], FS_EDA_WRIST, 'w_EDA'))
    if 'TEMP' in wrist_data and len(wrist_data['TEMP']) > 0:
        all_features.update(compute_temp_features(wrist_data['TEMP'], FS_TEMP_WRIST, 'w_TEMP'))
    if 'ACC' in wrist_data and len(wrist_data['ACC']) > 0:
        all_features.update(compute_acc_features(wrist_data['ACC'], FS_ACC_WRIST, 'w_ACC'))
    
    # Etiqueta por voto mayoritario
    label_mode = stats.mode(labels_window, keepdims=True)[0][0]
    all_features['label'] = label_mode
    
    return all_features


def extract_features_subject(data, subject_id):
    """
    Extrae características de todas las ventanas de un sujeto.
    Solo procesa etiquetas de interés: 1=baseline, 2=stress, 3=amusement.
    """
    labels = data['label'].flatten()
    chest_signals = data['signal']['chest']
    wrist_signals = data['signal']['wrist']
    
    chest_window = WINDOW_SIZE * FS_CHEST
    chest_shift = WINDOW_SHIFT * FS_CHEST
    n_samples_chest = len(labels)
    all_features_list = []
    
    start = 0
    window_count = 0
    while start + chest_window <= n_samples_chest:
        end = start + chest_window
        labels_window = labels[start:end]
        
        # Verificar etiquetas válidas (al menos 80% de la ventana)
        valid_labels = labels_window[
            (labels_window == 1) | (labels_window == 2) | (labels_window == 3)
        ]
        if len(valid_labels) < 0.8 * len(labels_window):
            start += chest_shift
            continue
        
        # Extraer señales chest
        chest_window_data = {}
        for key in chest_signals:
            chest_window_data[key] = chest_signals[key][start:end]
        
        # Índices correspondientes para wrist (diferente frecuencia de muestreo)
        time_start = start / FS_CHEST
        time_end = end / FS_CHEST
        
        wrist_window_data = {}
        for key in wrist_signals:
            sig = wrist_signals[key]
            if key == 'BVP':
                ws, we = int(time_start * FS_BVP), int(time_end * FS_BVP)
            elif key in ['EDA', 'TEMP']:
                ws, we = int(time_start * FS_EDA_WRIST), int(time_end * FS_EDA_WRIST)
            elif key == 'ACC':
                ws, we = int(time_start * FS_ACC_WRIST), int(time_end * FS_ACC_WRIST)
            else:
                ws, we = 0, 0
            wrist_window_data[key] = sig[ws:we] if we <= len(sig) else sig[ws:]
        
        features = extract_features_window(chest_window_data, wrist_window_data, labels_window)
        features['subject_id'] = subject_id
        all_features_list.append(features)
        window_count += 1
        start += chest_shift
    
    print(f'  Sujeto S{subject_id}: {window_count} ventanas procesadas')
    return all_features_list

print('Funciones de extracción por ventana y sujeto definidas.')

Funciones de extracción por ventana y sujeto definidas.


### 1.5 Generador de Datos Sintéticos (Fallback)

Si el dataset WESAD no está disponible localmente, se generan datos sintéticos que replican la estructura y los rangos fisiológicos reales para poder ejecutar todo el pipeline de demostración.

In [7]:
def generate_synthetic_wesad_data():
    """
    Genera datos sintéticos que simulan la estructura del dataset WESAD.
    Los valores están basados en rangos fisiológicos reales:
    - ECG: HR 60-100 bpm, HRV reducida bajo estrés
    - EDA: 0.01-20 μS, estrés causa aumento
    - Temperatura: 30-37°C
    - Respiración: tasa elevada bajo estrés
    """
    print('\n*** MODO SINTÉTICO: Generando datos que simulan la estructura WESAD ***')
    print('*** Para usar datos reales, descarga el dataset y ajusta DATA_PATH ***\n')
    
    np.random.seed(42)
    all_features = []
    
    for sid in SUBJECT_IDS:
        subject_offset = np.random.normal(0, 0.1)
        n_baseline = np.random.randint(12, 18)
        n_stress = np.random.randint(8, 14)
        n_amusement = np.random.randint(8, 12)
        
        for condition, n_windows, label in [
            ('baseline', n_baseline, 1),
            ('stress', n_stress, 2),
            ('amusement', n_amusement, 3)
        ]:
            for _ in range(n_windows):
                feat = {}
                noise = np.random.normal(0, 0.05)
                
                # ECG features
                hr_base = (95 if condition == 'stress' else 72) + np.random.normal(0, 8)
                hrv_base = 0.03 if condition == 'stress' else 0.06
                
                for k, (mu, sig) in {
                    'c_ECG_mean': (0.02, 0.005), 'c_ECG_std': (0.15, 0.02),
                    'c_ECG_min': (-0.5, 0.1), 'c_ECG_max': (1.0, 0.2),
                    'c_ECG_median': (0.01, 0.005), 'c_ECG_kurtosis': (3.0, 1),
                    'c_ECG_skewness': (0.5, 0.3), 'c_ECG_q25': (-0.05, 0.01),
                    'c_ECG_q75': (0.08, 0.01), 'c_ECG_rms': (0.15, 0.02),
                    'c_ECG_zcr': (0.1, 0.02), 'c_ECG_peak_freq': (1.2, 0.2),
                    'c_ECG_spectral_energy': (0.005, 0.001),
                    'c_ECG_spectral_entropy': (5.0, 0.5),
                }.items():
                    feat[k] = mu + np.random.normal(0, sig) + subject_offset * 0.01
                
                feat['c_ECG_range'] = feat['c_ECG_max'] - feat['c_ECG_min']
                feat['c_ECG_iqr'] = feat['c_ECG_q75'] - feat['c_ECG_q25']
                feat['c_ECG_hr_mean'] = hr_base + subject_offset * 5
                feat['c_ECG_hr_std'] = 5.0 + np.random.normal(0, 1.5)
                feat['c_ECG_rr_mean'] = 60.0 / hr_base
                feat['c_ECG_rr_std'] = hrv_base + np.random.normal(0, 0.01)
                feat['c_ECG_rmssd'] = hrv_base * 1.2 + np.random.normal(0, 0.005)
                feat['c_ECG_pnn50'] = (0.08 if condition == 'stress' else 0.2) + np.random.normal(0, 0.03)
                
                # EDA features
                eda_base = (8.0 if condition == 'stress' else 3.0) + np.random.normal(0, 1.5)
                scr_peaks = np.random.randint(5, 15) if condition == 'stress' else np.random.randint(0, 5)
                
                feat['c_EDA_mean'] = eda_base + subject_offset
                feat['c_EDA_std'] = eda_base * 0.2 + np.random.normal(0, 0.3)
                feat['c_EDA_min'] = eda_base * 0.5 + np.random.normal(0, 0.2)
                feat['c_EDA_max'] = eda_base * 1.5 + np.random.normal(0, 0.5)
                feat['c_EDA_median'] = eda_base + np.random.normal(0, 0.2)
                feat['c_EDA_range'] = feat['c_EDA_max'] - feat['c_EDA_min']
                feat['c_EDA_kurtosis'] = 2.5 + np.random.normal(0, 1)
                feat['c_EDA_skewness'] = 0.3 + np.random.normal(0, 0.2)
                feat['c_EDA_q25'] = eda_base * 0.8 + np.random.normal(0, 0.2)
                feat['c_EDA_q75'] = eda_base * 1.2 + np.random.normal(0, 0.2)
                feat['c_EDA_iqr'] = feat['c_EDA_q75'] - feat['c_EDA_q25']
                feat['c_EDA_rms'] = eda_base * 1.05 + np.random.normal(0, 0.3)
                feat['c_EDA_zcr'] = 0.01 + np.random.normal(0, 0.005)
                feat['c_EDA_peak_freq'] = 0.05 + np.random.normal(0, 0.02)
                feat['c_EDA_spectral_energy'] = eda_base**2 * 0.01 + np.random.normal(0, 0.01)
                feat['c_EDA_spectral_entropy'] = 4.0 + np.random.normal(0, 0.5)
                feat['c_EDA_scl_mean'] = eda_base * 0.9 + np.random.normal(0, 0.2)
                feat['c_EDA_scl_std'] = 0.3 + np.random.normal(0, 0.1)
                feat['c_EDA_scr_mean'] = 0.1 + np.random.normal(0, 0.05)
                feat['c_EDA_scr_std'] = 0.5 + np.random.normal(0, 0.1)
                feat['c_EDA_scr_num_peaks'] = scr_peaks
                feat['c_EDA_deriv_mean'] = 0.001 + np.random.normal(0, 0.0005)
                feat['c_EDA_deriv_std'] = 0.01 + np.random.normal(0, 0.003)
                
                # EMG features
                emg_base = (0.05 if condition == 'stress' else 0.02) + np.random.normal(0, 0.01)
                feat['c_EMG_mean'] = emg_base + noise
                feat['c_EMG_std'] = emg_base * 2.0 + np.random.normal(0, 0.01)
                feat['c_EMG_min'] = -emg_base * 5 + np.random.normal(0, 0.02)
                feat['c_EMG_max'] = emg_base * 5 + np.random.normal(0, 0.02)
                feat['c_EMG_median'] = emg_base * 0.1 + np.random.normal(0, 0.005)
                feat['c_EMG_range'] = feat['c_EMG_max'] - feat['c_EMG_min']
                feat['c_EMG_kurtosis'] = 5.0 + np.random.normal(0, 2)
                feat['c_EMG_skewness'] = 0.1 + np.random.normal(0, 0.3)
                feat['c_EMG_q25'] = -emg_base + np.random.normal(0, 0.005)
                feat['c_EMG_q75'] = emg_base + np.random.normal(0, 0.005)
                feat['c_EMG_iqr'] = feat['c_EMG_q75'] - feat['c_EMG_q25']
                feat['c_EMG_rms'] = emg_base * 1.5 + np.random.normal(0, 0.005)
                feat['c_EMG_zcr'] = 0.4 + np.random.normal(0, 0.05)
                feat['c_EMG_peak_freq'] = 50 + np.random.normal(0, 15)
                feat['c_EMG_spectral_energy'] = emg_base**2 + np.random.normal(0, 0.001)
                feat['c_EMG_spectral_entropy'] = 6.0 + np.random.normal(0, 0.5)
                feat['c_EMG_mav'] = abs(emg_base) + np.random.normal(0, 0.005)
                feat['c_EMG_var'] = emg_base**2 * 3 + np.random.normal(0, 0.001)
                feat['c_EMG_wl'] = emg_base * 500 + np.random.normal(0, 20)
                
                # Respiración features
                resp_rate = (22 if condition == 'stress' else 16) + np.random.normal(0, 2)
                feat['c_RESP_mean'] = np.random.normal(0, 0.1)
                feat['c_RESP_std'] = 200 + np.random.normal(0, 50)
                feat['c_RESP_min'] = -500 + np.random.normal(0, 100)
                feat['c_RESP_max'] = 500 + np.random.normal(0, 100)
                feat['c_RESP_median'] = np.random.normal(0, 0.1)
                feat['c_RESP_range'] = feat['c_RESP_max'] - feat['c_RESP_min']
                feat['c_RESP_kurtosis'] = 2.0 + np.random.normal(0, 0.5)
                feat['c_RESP_skewness'] = np.random.normal(0, 0.2)
                feat['c_RESP_q25'] = -150 + np.random.normal(0, 30)
                feat['c_RESP_q75'] = 150 + np.random.normal(0, 30)
                feat['c_RESP_iqr'] = feat['c_RESP_q75'] - feat['c_RESP_q25']
                feat['c_RESP_rms'] = 200 + np.random.normal(0, 50)
                feat['c_RESP_zcr'] = 0.005 + np.random.normal(0, 0.001)
                feat['c_RESP_peak_freq'] = resp_rate / 60 + np.random.normal(0, 0.02)
                feat['c_RESP_spectral_energy'] = 10000 + np.random.normal(0, 2000)
                feat['c_RESP_spectral_entropy'] = 4.5 + np.random.normal(0, 0.5)
                feat['c_RESP_rate_mean'] = resp_rate
                feat['c_RESP_rate_std'] = 2.0 + np.random.normal(0, 0.5)
                feat['c_RESP_insp_time'] = 60.0 / resp_rate + np.random.normal(0, 0.2)
                
                # Temperatura chest
                temp_base = (34.5 if condition == 'stress' else 35.0) + np.random.normal(0, 0.3)
                feat['c_TEMP_mean'] = temp_base + subject_offset * 0.5
                feat['c_TEMP_std'] = 0.05 + np.random.normal(0, 0.01)
                feat['c_TEMP_min'] = temp_base - 0.1 + np.random.normal(0, 0.02)
                feat['c_TEMP_max'] = temp_base + 0.1 + np.random.normal(0, 0.02)
                feat['c_TEMP_median'] = temp_base + np.random.normal(0, 0.02)
                feat['c_TEMP_range'] = feat['c_TEMP_max'] - feat['c_TEMP_min']
                feat['c_TEMP_kurtosis'] = 2.0 + np.random.normal(0, 0.5)
                feat['c_TEMP_skewness'] = np.random.normal(0, 0.1)
                feat['c_TEMP_q25'] = temp_base - 0.03 + np.random.normal(0, 0.01)
                feat['c_TEMP_q75'] = temp_base + 0.03 + np.random.normal(0, 0.01)
                feat['c_TEMP_iqr'] = feat['c_TEMP_q75'] - feat['c_TEMP_q25']
                feat['c_TEMP_rms'] = temp_base + np.random.normal(0, 0.02)
                feat['c_TEMP_zcr'] = 0.001 + np.random.normal(0, 0.0005)
                feat['c_TEMP_slope'] = 0.0001 + np.random.normal(0, 0.00005)
                feat['c_TEMP_deriv_mean'] = 0.0001 + np.random.normal(0, 0.0001)
                feat['c_TEMP_deriv_std'] = 0.001 + np.random.normal(0, 0.0003)
                
                # ACC chest
                feat['c_ACC_x_mean'] = 0.9 + np.random.normal(0, 0.05)
                feat['c_ACC_x_std'] = 0.05 + np.random.normal(0, 0.01)
                feat['c_ACC_y_mean'] = -0.2 + np.random.normal(0, 0.05)
                feat['c_ACC_y_std'] = 0.04 + np.random.normal(0, 0.01)
                feat['c_ACC_z_mean'] = -0.3 + np.random.normal(0, 0.05)
                feat['c_ACC_z_std'] = 0.04 + np.random.normal(0, 0.01)
                feat['c_ACC_mag_mean'] = 1.0 + np.random.normal(0, 0.03)
                feat['c_ACC_mag_std'] = 0.05 + np.random.normal(0, 0.01)
                feat['c_ACC_mag_peak_freq'] = 0.5 + np.random.normal(0, 0.2)
                feat['c_ACC_mag_spectral_energy'] = 0.01 + np.random.normal(0, 0.003)
                feat['c_ACC_mag_spectral_entropy'] = 3.0 + np.random.normal(0, 0.5)
                
                # BVP wrist
                feat['w_BVP_mean'] = np.random.normal(0, 0.5)
                feat['w_BVP_std'] = 50 + np.random.normal(0, 15)
                feat['w_BVP_min'] = -150 + np.random.normal(0, 40)
                feat['w_BVP_max'] = 150 + np.random.normal(0, 40)
                feat['w_BVP_median'] = np.random.normal(0, 0.5)
                feat['w_BVP_range'] = feat['w_BVP_max'] - feat['w_BVP_min']
                feat['w_BVP_kurtosis'] = 2.0 + np.random.normal(0, 0.8)
                feat['w_BVP_skewness'] = np.random.normal(0, 0.3)
                feat['w_BVP_q25'] = -30 + np.random.normal(0, 10)
                feat['w_BVP_q75'] = 30 + np.random.normal(0, 10)
                feat['w_BVP_iqr'] = feat['w_BVP_q75'] - feat['w_BVP_q25']
                feat['w_BVP_rms'] = 50 + np.random.normal(0, 15)
                feat['w_BVP_zcr'] = 0.1 + np.random.normal(0, 0.02)
                feat['w_BVP_peak_freq'] = hr_base / 60 + np.random.normal(0, 0.05)
                feat['w_BVP_spectral_energy'] = 5000 + np.random.normal(0, 1500)
                feat['w_BVP_spectral_entropy'] = 4.0 + np.random.normal(0, 0.5)
                
                # EDA wrist
                w_eda = eda_base * 0.6 + np.random.normal(0, 0.3)
                feat['w_EDA_mean'] = w_eda
                feat['w_EDA_std'] = w_eda * 0.15 + np.random.normal(0, 0.1)
                feat['w_EDA_min'] = w_eda * 0.6 + np.random.normal(0, 0.1)
                feat['w_EDA_max'] = w_eda * 1.4 + np.random.normal(0, 0.2)
                feat['w_EDA_median'] = w_eda + np.random.normal(0, 0.1)
                feat['w_EDA_range'] = feat['w_EDA_max'] - feat['w_EDA_min']
                feat['w_EDA_kurtosis'] = 2.5 + np.random.normal(0, 0.8)
                feat['w_EDA_skewness'] = 0.3 + np.random.normal(0, 0.2)
                feat['w_EDA_q25'] = w_eda * 0.85 + np.random.normal(0, 0.1)
                feat['w_EDA_q75'] = w_eda * 1.15 + np.random.normal(0, 0.1)
                feat['w_EDA_iqr'] = feat['w_EDA_q75'] - feat['w_EDA_q25']
                feat['w_EDA_rms'] = w_eda * 1.02 + np.random.normal(0, 0.1)
                feat['w_EDA_zcr'] = 0.05 + np.random.normal(0, 0.02)
                feat['w_EDA_peak_freq'] = 0.03 + np.random.normal(0, 0.01)
                feat['w_EDA_spectral_energy'] = w_eda**2 * 0.01 + np.random.normal(0, 0.01)
                feat['w_EDA_spectral_entropy'] = 3.5 + np.random.normal(0, 0.5)
                feat['w_EDA_scl_mean'] = w_eda * 0.95 + np.random.normal(0, 0.1)
                feat['w_EDA_scl_std'] = 0.15 + np.random.normal(0, 0.05)
                feat['w_EDA_scr_mean'] = 0.05 + np.random.normal(0, 0.02)
                feat['w_EDA_scr_std'] = 0.2 + np.random.normal(0, 0.05)
                feat['w_EDA_scr_num_peaks'] = max(0, scr_peaks - np.random.randint(0, 3))
                feat['w_EDA_deriv_mean'] = 0.001 + np.random.normal(0, 0.0005)
                feat['w_EDA_deriv_std'] = 0.005 + np.random.normal(0, 0.002)
                
                # TEMP wrist
                wt = temp_base - 2 + np.random.normal(0, 0.3)
                feat['w_TEMP_mean'] = wt
                feat['w_TEMP_std'] = 0.03 + np.random.normal(0, 0.01)
                feat['w_TEMP_min'] = wt - 0.08 + np.random.normal(0, 0.02)
                feat['w_TEMP_max'] = wt + 0.08 + np.random.normal(0, 0.02)
                feat['w_TEMP_median'] = wt + np.random.normal(0, 0.02)
                feat['w_TEMP_range'] = feat['w_TEMP_max'] - feat['w_TEMP_min']
                feat['w_TEMP_kurtosis'] = 2.0 + np.random.normal(0, 0.5)
                feat['w_TEMP_skewness'] = np.random.normal(0, 0.1)
                feat['w_TEMP_q25'] = wt - 0.02 + np.random.normal(0, 0.01)
                feat['w_TEMP_q75'] = wt + 0.02 + np.random.normal(0, 0.01)
                feat['w_TEMP_iqr'] = feat['w_TEMP_q75'] - feat['w_TEMP_q25']
                feat['w_TEMP_rms'] = wt + np.random.normal(0, 0.02)
                feat['w_TEMP_zcr'] = 0.001 + np.random.normal(0, 0.0005)
                feat['w_TEMP_slope'] = 0.0001 + np.random.normal(0, 0.00005)
                feat['w_TEMP_deriv_mean'] = 0.0001 + np.random.normal(0, 0.0001)
                feat['w_TEMP_deriv_std'] = 0.0005 + np.random.normal(0, 0.0002)
                
                # ACC wrist
                feat['w_ACC_x_mean'] = np.random.normal(0, 0.1)
                feat['w_ACC_x_std'] = 0.1 + np.random.normal(0, 0.03)
                feat['w_ACC_y_mean'] = -0.5 + np.random.normal(0, 0.1)
                feat['w_ACC_y_std'] = 0.1 + np.random.normal(0, 0.03)
                feat['w_ACC_z_mean'] = -0.8 + np.random.normal(0, 0.1)
                feat['w_ACC_z_std'] = 0.08 + np.random.normal(0, 0.02)
                feat['w_ACC_mag_mean'] = 1.0 + np.random.normal(0, 0.05)
                feat['w_ACC_mag_std'] = 0.1 + np.random.normal(0, 0.03)
                feat['w_ACC_mag_peak_freq'] = 0.5 + np.random.normal(0, 0.2)
                feat['w_ACC_mag_spectral_energy'] = 0.02 + np.random.normal(0, 0.005)
                feat['w_ACC_mag_spectral_entropy'] = 3.0 + np.random.normal(0, 0.5)
                
                feat['label'] = label
                feat['subject_id'] = sid
                all_features.append(feat)
    
    return pd.DataFrame(all_features)

print('Función de datos sintéticos definida.')

Función de datos sintéticos definida.


### 1.6 Ejecución de la Extracción de Características

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EJECUCIÓN DE LA EXTRACCIÓN
# ═══════════════════════════════════════════════════════════════════
print('Intentando cargar el dataset WESAD...')

try:
    test_path = os.path.join(DATA_PATH, 'S2', 'S2.pkl')
    if not os.path.exists(test_path):
        raise FileNotFoundError(f'No se encontró {test_path}')
    
    print('Dataset WESAD encontrado. Procesando datos reales...')
    all_features_list = []
    
    for sid in SUBJECT_IDS:
        print(f'\nProcesando sujeto S{sid}...')
        try:
            data = load_subject_data(DATA_PATH, sid)
            features = extract_features_subject(data, sid)
            all_features_list.extend(features)
        except Exception as e:
            print(f'  Error con sujeto S{sid}: {e}')
    
    df_features = pd.DataFrame(all_features_list)
    USE_SYNTHETIC = False

except FileNotFoundError:
    print('Dataset WESAD no encontrado localmente.')
    df_features = generate_synthetic_wesad_data()
    USE_SYNTHETIC = True

# Crear etiqueta binaria: stress(2) vs no-stress (baseline=1, amusement=3)
df_features['label_binary'] = df_features['label'].map(LABEL_MAP)
df_features = df_features.dropna(subset=['label_binary'])
df_features['label_binary'] = df_features['label_binary'].astype(int)

# Guardar dataset
df_features.to_csv(os.path.join(OUTPUT_DIR, 'wesad_features.csv'), index=False)

# Separar features y etiquetas
feature_cols = [c for c in df_features.columns if c not in ['label', 'label_binary', 'subject_id']]
X = df_features[feature_cols].copy()
y = df_features['label_binary'].copy()

print(f'\n{"="*60}')
print(f'RESUMEN DE LA EXTRACCIÓN DE CARACTERÍSTICAS')
print(f'{"="*60}')
print(f'Total de muestras (ventanas):  {len(df_features)}')
print(f'Total de características:       {len(feature_cols)}')
print(f'Sujetos procesados:            {df_features["subject_id"].nunique()}')
print(f'\nDistribución de clases originales:')
print(df_features['label'].value_counts().to_string())
print(f'\nDistribución binaria (0=no-stress, 1=stress):')
print(y.value_counts().to_string())

Intentando cargar el dataset WESAD...
Dataset WESAD encontrado. Procesando datos reales...

Procesando sujeto S2...


---
## Actividad 2: Análisis Exploratorio de Datos (EDA)

Análisis exhaustivo del dataset de características extraído en la Actividad 1.

### 2.1 Información General del Dataset

In [ ]:
print(f'Shape del dataset: {df_features.shape}')
print(f'\nTipos de datos:')
print(X.dtypes.value_counts().to_string())
print(f'\nEstadísticas descriptivas (primeras 10 features):')
X[feature_cols[:10]].describe().round(4)

### 2.2 Análisis de Valores Faltantes e Infinitos

In [ ]:
# Valores faltantes
missing = X.isnull().sum()
missing_pct = (missing / len(X)) * 100
missing_summary = pd.DataFrame({'Faltantes': missing, 'Porcentaje': missing_pct})
missing_with_values = missing_summary[missing_summary['Faltantes'] > 0]

if len(missing_with_values) > 0:
    print(f'Características con valores faltantes ({len(missing_with_values)}):')
    print(missing_with_values.sort_values('Porcentaje', ascending=False).head(20))
else:
    print('✓ No hay valores faltantes en el dataset.')

# Valores infinitos
inf_count = np.isinf(X.select_dtypes(include=[np.number])).sum()
inf_cols = inf_count[inf_count > 0]
if len(inf_cols) > 0:
    print(f'\nCaracterísticas con valores infinitos:')
    print(inf_cols)
else:
    print('✓ No hay valores infinitos en el dataset.')

### 2.3 Distribución de Clases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clases originales
label_names = {1: 'Baseline', 2: 'Stress', 3: 'Amusement'}
orig_counts = df_features['label'].value_counts()
colors_orig = ['#3498db', '#e74c3c', '#2ecc71']
axes[0].bar([label_names.get(x, str(x)) for x in orig_counts.index],
            orig_counts.values, color=colors_orig[:len(orig_counts)])
axes[0].set_title('Distribución de Clases Originales', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de muestras')
for i, v in enumerate(orig_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Clases binarias
bin_counts = y.value_counts()
bin_names = {0: 'No-Stress', 1: 'Stress'}
colors_bin = ['#3498db', '#e74c3c']
axes[1].bar([bin_names[x] for x in bin_counts.index],
            bin_counts.values, color=colors_bin)
axes[1].set_title('Distribución Binaria', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Número de muestras')
for i, v in enumerate(bin_counts.values):
    axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig1_class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

# Evaluar desbalance
class_ratio = bin_counts.min() / bin_counts.max()
print(f'Ratio minoría/mayoría: {class_ratio:.3f}')
if class_ratio < 0.5:
    print('⚠️  Dataset DESBALANCEADO - considerar técnicas de balanceo')
else:
    print('✓  Dataset relativamente balanceado')

### 2.4 Análisis de Outliers

In [ ]:
outlier_counts = {}
for col in feature_cols:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = ((X[col] < Q1 - 1.5 * IQR) | (X[col] > Q3 + 1.5 * IQR)).sum()
    if n_outliers > 0:
        outlier_counts[col] = n_outliers

outlier_df = pd.DataFrame.from_dict(outlier_counts, orient='index', columns=['n_outliers'])
outlier_df['pct'] = (outlier_df['n_outliers'] / len(X) * 100).round(2)
outlier_df = outlier_df.sort_values('n_outliers', ascending=False)
print('Top 15 características con más outliers (método IQR):')
outlier_df.head(15)

### 2.5 Características Más Discriminativas (t-test)

In [ ]:
stress_data = X[y == 1]
no_stress_data = X[y == 0]

class_diffs = {}
for col in feature_cols:
    try:
        t_stat, p_val = stats.ttest_ind(stress_data[col].dropna(), no_stress_data[col].dropna())
        class_diffs[col] = {
            'media_stress': stress_data[col].mean(),
            'media_no_stress': no_stress_data[col].mean(),
            'dif_pct': abs(stress_data[col].mean() - no_stress_data[col].mean()) / (abs(no_stress_data[col].mean()) + 1e-10) * 100,
            't_stat': t_stat,
            'p_value': p_val
        }
    except Exception:
        pass

diff_df = pd.DataFrame(class_diffs).T.sort_values('p_value')
print('Top 15 características más discriminativas (menor p-value):')
diff_df[['media_stress', 'media_no_stress', 'dif_pct', 't_stat', 'p_value']].head(15).round(6)

### 2.6 Boxplots de Features Más Discriminativas

In [ ]:
top_features = diff_df.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    data_plot = pd.DataFrame({'value': X[feat], 'class': y.map({0: 'No-Stress', 1: 'Stress'})})
    sns.boxplot(data=data_plot, x='class', y='value', ax=axes[i], palette=['#3498db', '#e74c3c'])
    p_val = diff_df.loc[feat, 'p_value']
    axes[i].set_title(f'{feat}\n(p={p_val:.2e})', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Top 6 Características Más Discriminativas (Stress vs No-Stress)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_top_features_boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()

### 2.7 Matriz de Correlaciones

In [ ]:
top20_features = diff_df.head(20).index.tolist()
fig, ax = plt.subplots(figsize=(14, 12))
corr_sub = X[top20_features].corr()
mask = np.triu(np.ones_like(corr_sub, dtype=bool))
sns.heatmap(corr_sub, mask=mask, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', square=True, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8}, annot_kws={'size': 7})
ax.set_title('Matriz de Correlaciones - Top 20 Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# Contar pares altamente correlacionados
corr_matrix = X.corr()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(i, c, upper_tri.loc[i, c]) for c in upper_tri.columns for i in upper_tri.index if abs(upper_tri.loc[i, c]) > 0.95]
print(f'Pares con correlación > 0.95: {len(high_corr)}')

### 2.8 Distribuciones por Clase

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    for cls, color, label in [(0, '#3498db', 'No-Stress'), (1, '#e74c3c', 'Stress')]:
        axes[i].hist(X.loc[y == cls, feat].dropna(), bins=30, alpha=0.5,
                     color=color, label=label, density=True)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Densidad')

plt.suptitle('Distribuciones de Top Features por Clase', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig4_distributions_by_class.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Actividad 3: Preprocesamiento de Datos

Basado en las necesidades identificadas en el EDA:
1. Imputación de valores faltantes (mediana)
2. Tratamiento de valores infinitos
3. Eliminación de características con varianza cero
4. Eliminación de características altamente correlacionadas (>0.98)
5. Winsorización de outliers (percentiles 1-99)
6. Estandarización (StandardScaler)

In [ ]:
X_processed = X.copy()

# ═══ 3.1 Valores faltantes ═══
print('--- 3.1 Tratamiento de Valores Faltantes ---')
n_missing = X_processed.isnull().sum().sum()
print(f'Valores faltantes: {n_missing}')
for col in X_processed.columns:
    if X_processed[col].isnull().any():
        X_processed[col].fillna(X_processed[col].median(), inplace=True)

# ═══ 3.2 Valores infinitos ═══
print('\n--- 3.2 Tratamiento de Valores Infinitos ---')
n_inf = np.isinf(X_processed.select_dtypes(include=[np.number])).sum().sum()
print(f'Valores infinitos: {n_inf}')
X_processed.replace([np.inf, -np.inf], np.nan, inplace=True)
for col in X_processed.columns:
    if X_processed[col].isnull().any():
        X_processed[col].fillna(X_processed[col].median(), inplace=True)

# ═══ 3.3 Varianza cero ═══
print('\n--- 3.3 Eliminación de Características con Varianza Cero ---')
zero_var = X_processed.columns[X_processed.var() < 1e-10].tolist()
if zero_var:
    print(f'Eliminando {len(zero_var)} características')
    X_processed.drop(columns=zero_var, inplace=True)
else:
    print('Ninguna característica con varianza cero.')

# ═══ 3.4 Alta correlación ═══
print('\n--- 3.4 Eliminación por Alta Correlación (>0.98) ---')
corr_abs = X_processed.corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.98)]
print(f'Eliminando {len(to_drop)} características altamente correlacionadas')
X_processed.drop(columns=to_drop, inplace=True)

# ═══ 3.5 Winsorización ═══
print('\n--- 3.5 Winsorización de Outliers (percentiles 1-99) ---')
n_clipped = 0
for col in X_processed.columns:
    p1, p99 = X_processed[col].quantile(0.01), X_processed[col].quantile(0.99)
    n_clipped += ((X_processed[col] < p1) | (X_processed[col] > p99)).sum()
    X_processed[col] = X_processed[col].clip(p1, p99)
print(f'Valores winsorizados: {n_clipped}')

# ═══ 3.6 Estandarización ═══
print('\n--- 3.6 Estandarización (StandardScaler) ---')
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_processed),
    columns=X_processed.columns,
    index=X_processed.index
)
print(f'Media después de escalar: {X_scaled.mean().mean():.6f}')
print(f'Std después de escalar:   {X_scaled.std().mean():.6f}')

print(f'\n{"="*60}')
print(f'RESUMEN DEL PREPROCESAMIENTO')
print(f'{"="*60}')
print(f'Características originales:  {len(feature_cols)}')
print(f'Características finales:     {X_scaled.shape[1]}')
print(f'Muestras:                    {len(X_scaled)}')

---
## Actividad 4: Ranking de Características con Factor de Fisher

El **Factor de Fisher** (Fisher Discriminant Ratio) mide la capacidad discriminativa de cada característica individual:

$$F(f) = \frac{(\mu_1 - \mu_2)^2}{\sigma_1^2 + \sigma_2^2}$$

Donde:
- $\mu_1, \mu_2$: medias de la característica para cada clase
- $\sigma_1^2, \sigma_2^2$: varianzas de la característica para cada clase

**Interpretación:** Un valor mayor indica mejor separación entre clases → más útil para detectar estrés.

In [ ]:
def fisher_score(X_data, y_data):
    """
    Calcula el Factor de Fisher para cada característica.
    
    Para clasificación binaria:
    F(f) = (μ₁ - μ₂)² / (σ₁² + σ₂²)
    """
    classes = np.unique(y_data)
    n_features = X_data.shape[1]
    scores = np.zeros(n_features)
    
    for i in range(n_features):
        feat_vals = X_data.iloc[:, i] if hasattr(X_data, 'iloc') else X_data[:, i]
        means, variances = [], []
        
        for c in classes:
            mask = y_data == c
            class_vals = feat_vals[mask]
            means.append(np.mean(class_vals))
            variances.append(np.var(class_vals))
        
        numerator = (means[0] - means[1]) ** 2
        denominator = variances[0] + variances[1]
        scores[i] = numerator / denominator if denominator > 1e-10 else 0
    
    return scores


# Calcular Fisher Score
print('Calculando Factor de Fisher para cada característica...')
fisher_scores = fisher_score(X_scaled, y)

fisher_df = pd.DataFrame({
    'Característica': X_scaled.columns,
    'Fisher_Score': fisher_scores
}).sort_values('Fisher_Score', ascending=False).reset_index(drop=True)
fisher_df['Rank'] = range(1, len(fisher_df) + 1)

# Guardar ranking
fisher_df.to_csv(os.path.join(OUTPUT_DIR, 'fisher_ranking.csv'), index=False)

print(f'\nTop 20 características por Factor de Fisher:')
fisher_df[['Rank', 'Característica', 'Fisher_Score']].head(20)

### Visualización del Ranking de Fisher

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
top_n = min(25, len(fisher_df))
top_fisher = fisher_df.head(top_n)

colors = plt.cm.RdYlGn_r(np.linspace(0, 1, top_n))
bars = ax.barh(range(top_n), top_fisher['Fisher_Score'].values, color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_fisher['Característica'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Fisher Score', fontsize=12)
ax.set_title(f'Top {top_n} Características por Factor de Fisher\n(Mayor = Más Discriminativa)',
             fontsize=14, fontweight='bold')

for bar, val in zip(bars, top_fisher['Fisher_Score'].values):
    ax.text(bar.get_width() + max(top_fisher['Fisher_Score']) * 0.01,
            bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig6_fisher_ranking.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Actividad 5: Selección de las 5 Mejores Características
### Selección Escalar Hacia Adelante (Sequential Forward Selection - SFS)

**Algoritmo:**
1. Empezar con un conjunto vacío de características seleccionadas
2. En cada iteración, evaluar CADA característica candidata añadiéndola al conjunto y midiendo el accuracy con validación cruzada estratificada (5-fold)
3. Seleccionar la que maximice el accuracy
4. Repetir hasta tener 5 características

**Clasificador:** KNN (k=5)  
**Métrica:** Accuracy con validación cruzada estratificada

In [ ]:
def sequential_forward_selection(X_data, y_data, n_features_to_select=5,
                                  classifier=None, cv_folds=5):
    """
    Selección Escalar Hacia Adelante (SFS).
    
    En cada paso selecciona la característica que, añadida al conjunto actual,
    maximiza el accuracy en validación cruzada estratificada.
    """
    if classifier is None:
        classifier = KNeighborsClassifier(n_neighbors=5)
    
    available = list(X_data.columns)
    selected = []
    history = []
    
    print(f'Iniciando SFS: {len(available)} candidatas → seleccionar {n_features_to_select}')
    print(f'Clasificador: {classifier.__class__.__name__}, CV: {cv_folds} folds\n')
    
    for step in range(n_features_to_select):
        print(f'--- Paso {step + 1}/{n_features_to_select} ---')
        best_score = -1
        best_feat = None
        step_results = []
        
        for feat in available:
            candidate = selected + [feat]
            cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
            scores = cross_val_score(classifier, X_data[candidate], y_data, cv=cv, scoring='accuracy')
            mean_score = scores.mean()
            step_results.append({'feature': feat, 'accuracy': mean_score, 'std': scores.std()})
            
            if mean_score > best_score:
                best_score = mean_score
                best_feat = feat
        
        selected.append(best_feat)
        available.remove(best_feat)
        history.append({
            'step': step + 1,
            'feature_added': best_feat,
            'accuracy': best_score,
            'selected': selected.copy()
        })
        
        print(f'  Seleccionada: {best_feat}')
        print(f'  Accuracy: {best_score:.4f}')
        
        # Top 5 candidatas
        top5 = sorted(step_results, key=lambda x: -x['accuracy'])[:5]
        print(f'  Top 5 evaluadas:')
        for j, r in enumerate(top5):
            mark = ' ✓' if r['feature'] == best_feat else ''
            print(f'    {j+1}. {r["feature"]}: {r["accuracy"]:.4f} ± {r["std"]:.4f}{mark}')
        print()
    
    return selected, history

print('Función SFS definida.')

### Ejecución de la Selección Hacia Adelante

In [ ]:
# Ejecutar SFS
selected_features, sfs_history = sequential_forward_selection(
    X_scaled, y,
    n_features_to_select=5,
    classifier=KNeighborsClassifier(n_neighbors=5),
    cv_folds=5
)

### Resultados de la Selección

In [ ]:
print(f'{"="*70}')
print(f'5 MEJORES CARACTERÍSTICAS POR SELECCIÓN HACIA ADELANTE')
print(f'{"="*70}')

print(f'\n{"Paso":<6}{"Característica Añadida":<40}{"Accuracy":<15}')
print('-' * 61)
for entry in sfs_history:
    print(f'{entry["step"]:<6}{entry["feature_added"]:<40}{entry["accuracy"]:.4f}')

print(f'\n5 Características Seleccionadas:')
for i, feat in enumerate(selected_features, 1):
    fv = fisher_df[fisher_df['Característica'] == feat]['Fisher_Score'].values
    fs = f'{fv[0]:.4f}' if len(fv) > 0 else 'N/A'
    print(f'  {i}. {feat}  (Fisher Score: {fs})')

### Evaluación Comparativa

In [ ]:
print(f'{"="*70}')
print(f'EVALUACIÓN COMPARATIVA')
print(f'{"="*70}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for clf_name, clf in [
    ('KNN (k=5)', KNeighborsClassifier(n_neighbors=5)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42))
]:
    scores_sel = cross_val_score(clf, X_scaled[selected_features], y, cv=cv, scoring='accuracy')
    scores_all = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy')
    print(f'\n{clf_name}:')
    print(f'  Con 5 features seleccionadas: {scores_sel.mean():.4f} ± {scores_sel.std():.4f}')
    print(f'  Con todas ({X_scaled.shape[1]}) features:    {scores_all.mean():.4f} ± {scores_all.std():.4f}')

### Visualización de Resultados del SFS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Evolución del accuracy
steps = [h['step'] for h in sfs_history]
accuracies = [h['accuracy'] for h in sfs_history]
axes[0].plot(steps, accuracies, 'bo-', linewidth=2, markersize=10)
for s, a in zip(steps, accuracies):
    axes[0].annotate(f'{a:.4f}', (s, a), textcoords='offset points',
                     xytext=(0, 12), ha='center', fontsize=9, fontweight='bold')
axes[0].set_xlabel('Número de Características', fontsize=12)
axes[0].set_ylabel('Accuracy (CV)', fontsize=12)
axes[0].set_title('Evolución del Accuracy en SFS', fontsize=14, fontweight='bold')
axes[0].set_xticks(steps)
axes[0].grid(True, alpha=0.3)

# Fisher Score de las seleccionadas
fisher_sel = []
for feat in selected_features:
    fv = fisher_df[fisher_df['Característica'] == feat]['Fisher_Score'].values
    fisher_sel.append(fv[0] if len(fv) > 0 else 0)

colors = plt.cm.viridis(np.linspace(0.2, 0.8, 5))
bars = axes[1].barh(range(5), fisher_sel, color=colors)
axes[1].set_yticks(range(5))
axes[1].set_yticklabels(selected_features, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel('Fisher Score', fontsize=12)
axes[1].set_title('Fisher Score de las 5 Features\nSeleccionadas por SFS',
                   fontsize=14, fontweight='bold')
for bar, val in zip(bars, fisher_sel):
    axes[1].text(bar.get_width() + max(fisher_sel) * 0.02,
                 bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig7_sfs_results.png'), dpi=150, bbox_inches='tight')
plt.show()

### Guardar Resultados Finales

In [ ]:
# Guardar dataset final con las 5 mejores features
X_final = X_scaled[selected_features].copy()
X_final['label'] = y.values
X_final.to_csv(os.path.join(OUTPUT_DIR, 'wesad_final_5features.csv'), index=False)

# Guardar resumen JSON
results_summary = {
    'selected_features': selected_features,
    'sfs_history': [{'step': h['step'], 'feature_added': h['feature_added'],
                     'accuracy': h['accuracy']} for h in sfs_history],
    'fisher_ranking_top20': fisher_df.head(20).to_dict('records')
}
with open(os.path.join(OUTPUT_DIR, 'results_summary.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)

print('Archivos generados:')
print(f'  - {OUTPUT_DIR}wesad_features.csv           (dataset completo)')
print(f'  - {OUTPUT_DIR}wesad_final_5features.csv    (5 mejores features)')
print(f'  - {OUTPUT_DIR}fisher_ranking.csv           (ranking de Fisher)')
print(f'  - {OUTPUT_DIR}results_summary.json         (resumen)')
print(f'  - Gráficas fig1-fig7 en {OUTPUT_DIR}')

if USE_SYNTHETIC:
    print(f'\n⚠️  Se usaron datos SINTÉTICOS. Para datos reales:')
    print(f'   1. Descargar: https://www.kaggle.com/datasets/orvile/wesad-wearable-stress-affect-detection-dataset')
    print(f'   2. Extraer en: {DATA_PATH}')
    print(f'   3. Ejecutar nuevamente este notebook.')

---
## Conclusiones

Este notebook implementó el pipeline completo de ciencia de datos para detección de estrés usando el dataset WESAD:

1. **Extracción de características:** Se extrajeron 176+ features de señales fisiológicas multimodales (ECG, EDA, EMG, respiración, temperatura, acelerómetro) tanto de dispositivos de pecho como de muñeca.

2. **Análisis exploratorio:** Se identificó desbalance de clases, alta colinealidad entre features redundantes, y las señales EDA y HRV como las más discriminativas para estrés.

3. **Preprocesamiento:** Se aplicó imputación, eliminación de features redundantes, winsorización de outliers y estandarización.

4. **Factor de Fisher:** Permitió rankear las características según su capacidad discriminativa individual. Las features de HRV (RMSSD) y EDA resultaron las más informativas.

5. **Selección Hacia Adelante (SFS):** Se seleccionaron las 5 mejores características que, combinadas, maximizan el accuracy del clasificador.

---
*Pipeline basado en el paper de Schmidt et al. (2018) y el notebook de Kaggle de referencia.*